# Manual CCHE1D network (no TopAGNPS)

Build a small channel network from a hand-written definition, **independently of
TopAGNPS output files**. Every reach uses a **parabolic cross section** with a
constant Manning n and a **constant slope**, so the cross-section thalweg is
adjusted as we go upstream (each node's section is lifted by `slope * distance`
from the outlet, which sits at elevation 0).

Step by step:

1. the generic `Reach`/`Node` objects,
2. the parabolic cross section,
3. the Strahler order,
4. the thalweg adjustment (asserted: monotone upstream, outlet at 0),
5. the built CCHE1D network (node types, junction),
6. `validate_network` (must return no errors and no warnings),
7. writing the 8 files,
8. reading them back and re-validating (round trip),
9. inspecting the written `csec`/`csprf` rows (the parabolic profile).

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Make the package importable when running from a plain kernel.
# ROOT = Path.cwd()
ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import topagnps2cche1d as t2c
from topagnps2cche1d.network import read_cche1d_network, validate_network
from collections import Counter

print("package:", t2c.__file__)

package: /Users/luc/projects/topagnps2cche1d/topagnps2cche1d/__init__.py


In [19]:
# Topology (y increases downstream):
#
#   r1 (slope .0008)      r3 (slope .0006)
#        \                /
#         r2 (slope .0005) ---- r4 (slope .001, OUTLET)
#
# r1 and r3 merge into r2's head -> a 2-way JUNCTION.
# r2 drains into r4's head -> a trivial (1-way) connection.
J = (30.0, 60.0)   # junction pixel
D = (50.0, 100.0)  # r2 tail / r4 head pixel

def make_reach(rid, recv, slope, pts):
    r = t2c.Reach(id=rid, receiving_reach_id=recv, slope=slope)
    nodes = [t2c.Node(id=100*rid+k, x=x, y=y) for k, (x, y) in enumerate(pts)]
    for i, n in enumerate(nodes):
        n.dsid = nodes[i+1].id if i+1 < len(nodes) else None
        n.usid = nodes[i-1].id if i > 0 else None
        r.add_node(n)
    r.us_nd_id, r.ds_nd_id = nodes[0].id, nodes[-1].id
    return r

w = t2c.Watershed()
w.add_reach(make_reach(1, 2, 0.0008, [(0, 0), (15, 20), J]))
w.add_reach(make_reach(3, 2, 0.0006, [(30, 0), J]))
w.add_reach(make_reach(2, 4, 0.0005, [J, (40, 80), D]))
w.add_reach(make_reach(4, None, 0.0010, [D, (60, 140), (70, 160)]))
w.update_graph()
w.assign_strahler_number_to_reaches()

for reach_id, reach in w.reaches.items():
    print(f"Reach ID: {reach_id}, receiving reach:   {reach.receiving_reach_id}, strahler number: {reach.strahler_number}")

Reach ID: 1, receiving reach:   2, strahler number: 1
Reach ID: 3, receiving reach:   2, strahler number: 1
Reach ID: 2, receiving reach:   4, strahler number: 2
Reach ID: 4, receiving reach:   None, strahler number: 2


In [20]:
# Parabolic cross section (6 points, left -> right, looking downstream).
# ws = transverse distance from the channel centre line; zs = bed elevation.
# The two lowest points (around the -1 / +1 m bed) form the thalweg.
ws = [-14.0, -5.0, -1.0, 1.0, 5.0, 14.0]
zs = [6.0,   2.0,  0.0, 0.0, 2.0,  6.0]
w.assign_cross_section_to_all_points(
    cs_type="user_defined", ws=ws, zs=zs,
    lfp_idx=1, lbt_idx=2, rbt_idx=5, rfp_idx=6,
    mannings_roughness=0.035)
print("cross sections assigned:", len(w.cross_sections))

cross sections assigned: 11


In [21]:
# Adjust the thalweg: every section is lifted so the outlet sits at elevation 0
# and each node is higher than the one downstream by slope * distance.
w.adjust_cross_sections_elevation_with_slope()

def thalweg(node):
    return w.cross_sections[node.csid].get_thalweg_elevation()

outlet = w.reaches[4].nodes[w.reaches[4].ds_nd_id]
print("outlet thalweg (must be 0.0):", thalweg(outlet))
assert abs(thalweg(outlet)) < 1e-9

# Walk every reach from its US node to its DS node: the thalweg must
# strictly decrease in the downstream direction.
for rid, r in w.reaches.items():
    chain = []
    cur = r.us_nd_id
    while True:
        chain.append(thalweg(r.nodes[cur]))
        if cur == r.ds_nd_id:
            break
        cur = r.nodes[cur].dsid
    assert all(a > b - 1e-9 for a, b in zip(chain, chain[1:])), (rid, chain)
    print(f"reach {rid} thalweg US->DS:", [round(z, 4) for z in chain])
print("OK: outlet at 0, thalweg monotone upstream on every reach")

outlet thalweg (must be 0.0): 0.0
reach 1 thalweg US->DS: [np.float64(0.1401), np.float64(0.1201), np.float64(0.086)]
reach 3 thalweg US->DS: [np.float64(0.122), np.float64(0.086)]
reach 2 thalweg US->DS: [np.float64(0.086), np.float64(0.0748), np.float64(0.0636)]
reach 4 thalweg US->DS: [np.float64(0.0636), np.float64(0.0224), np.float64(0.0)]
OK: outlet at 0, thalweg monotone upstream on every reach


In [22]:
# Build the CCHE1D network and validate it.
net = w.build_cche1d_network(strahler_drop=0)   # validate=True -> raises on error
errors, warnings = validate_network(net)
print("nodes=%d channels=%d reaches=%d" % (len(net.nodes), len(net.channels), len(net.reaches)))
print("node types:", dict(sorted(Counter(n.nd_type for n in net.nodes).items())))
print("junctions:", [(j.node_id, j.branches345) for j in net.junctions])
assert errors == [] and warnings == []
print("validate_network: PASS (no errors, no warnings)")

nodes=11 channels=4 reaches=7
node types: {0: 2, 2: 1, 3: 3, 6: 4, 9: 1}
junctions: [(6, (-1, -1, -1))]
validate_network: PASS (no errors, no warnings)


In [ ]:
# Write all the .dat files.
out = Path.cwd().parent / "data_outputs" / "output_manual_parabolic"
out.mkdir(exist_ok=True)
paths = w.write_cche1d_dat_files(casename="manual", output_folder=str(out))
print("wrote:")
for k, p in sorted(paths.items()):
    print("  %-14s %6d bytes  %s" % (k, p.stat().st_size, p.name))

wrote:
  channel            53 bytes  manual_channel.dat
  csec              218 bytes  manual_csec.dat
  csprf            2013 bytes  manual_csprf.dat
  link              123 bytes  manual_link.dat
  nodes             428 bytes  manual_nodes.dat
  reach             179 bytes  manual_reach.dat
  tw_cell            46 bytes  manual_tw_cell.dat


In [25]:
# Round trip: read the files back and re-validate.
back = read_cche1d_network(out)
errors2, warnings2 = validate_network(back)
print("read back: nodes=%d channels=%d csecs=%d" % (len(back.nodes), len(back.channels), len(back.csecs)))
assert len(back.nodes) == len(net.nodes)
assert errors2 == [] and warnings2 == []
print("round-trip validate_network: PASS")

read back: nodes=11 channels=4 csecs=11
round-trip validate_network: PASS


In [26]:
# Inspect the written cross-section rows (this is the parabolic profile,
# lifted by the thalweg adjustment).
csec = (out/"manual_csec.dat").read_text().splitlines()
csprf = (out/"manual_csprf.dat").read_text().splitlines()
print("csec header count:", csec[0])
print("csec columns:", csec[1])
first_cs_id = csec[2].split("\t")[0]
rows = [l for l in csprf[2:] if l.split("\t")[1] == first_cs_id]
print(f"csprf points for CS {first_cs_id} ({len(rows)} rows):")
for l in rows:
    print("  " + l)
print("OK: parabolic profile written with the adjusted thalweg")

csec header count: 11
csec columns: CS_ID	CS_NPTS	CS_LOB	CS_LBT	CS_ROB	CS_RBT	CS_SVTYPE	CS_TYPE	CS_ORIGIN	CS_STATION
csprf points for CS 1 (6 rows):
  1	1	1	-14	6.1219524158	0.035	1
  2	1	2	-5	2.1219524158	0.035	1
  3	1	3	-1	0.1219524158	0.035	1
  4	1	4	1	0.1219524158	0.035	1
  5	1	5	5	2.1219524158	0.035	1
  6	1	6	14	6.1219524158	0.035	1
OK: parabolic profile written with the adjusted thalweg


## Result
A network built entirely from a manual definition (no raster, no TopAGNPS CSVs)
with a parabolic cross section and per-reach constant slopes validates cleanly
and round-trips through the file writer/reader — the same files the Fortran
model and the GUI import.